# S06 -- deduplikacija

**A kelias (be programavimo).** 18 bibliografinių įrašų iš kelių duomenų bazių; kai kurie yra to paties darbo dublikatai. Paleiskite 1-3 langelius ir žiūrėkite, kuriuos įrašus pašalina kaskada: pirma pagal DOI, paskui pagal panašų pavadinimą. Slankikliu keiskite panašumo slenkstį ir matykite, kaip tai keičia rezultatą.

**B kelias (programuotojams)** -- žemiau.

### Prieš pradedant

1. **Pirmas paleidimas.** Paleidus pirmą kodo langelį, Colab gali parodyti įspėjimą *„Notebook was not authored by Google“*. Tai normalu (sąsiuvinis atkeltas iš GitHub): spauskite **Run anyway**.
2. **Pirmas langelis** atsisiunčia kodą ir įdiegia bibliotekas (maždaug 1-2 min.). Paleiskite jį pirmą ir palaukite, kol pasirodys `Paruosta:`.
3. **Tikras DKM nebūtinas.** Jei norite jį išbandyti: kairėje Colab juostoje spauskite rakto piktogramą (*Secrets*), *Add new secret*, pavadinimas lygiai **`OPENAI_API_KEY`**, reikšmė -- jūsų OpenAI raktas, ir įjunkite *Notebook access*. Raktą rašykite tik čia, niekada į kodo langelį. Kreipiniai apmokami.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/rimraul/slr-automation.git"
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
if "google.colab" in sys.modules and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
for req in ['examples/chapter/requirements-colab.txt']:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit("pip install nepavyko (zr. klaida aukščiau): " + req)
print("Paruosta:", os.getcwd())

In [ ]:
#@title 1. Pasirinkite panašumo slenkstį ir DOI patikrą
THRESHOLD = 0.90  #@param {type:"slider", min:0.5, max:1.0, step:0.01}
DOI_TEST = "HTTP://DX.DOI.ORG/10.1000/Rev.001"  #@param {type:"string"}

In [ ]:
#@title 2. (nebūtina) Pridėkite savo įrašų
# Pridėkite savo įrašus (id, title, doi) -- jie bus deduplikuojami kartu su 18 pavyzdinių.
# Pavyzdys (atkomentuokite): vienas tikras dublikatas ir vienas panašus, bet skirtingas darbas.
EXTRA_RECORDS = [
    # {"id": "my-1", "title": "Large language models for title and abstract screening in systematic reviews", "doi": None},
    # {"id": "my-2", "title": "Large language models for full text screening in systematic reviews", "doi": None},
]

In [ ]:
#@title 3. Deduplikuoti
import difflib
from examples.chapter import toolkit
from examples.chapter.s06_deduplication import solution as s6

print("DOI normalizavimas:", repr(DOI_TEST), "->", repr(s6.normalise_doi(DOI_TEST)))
records = toolkit.load_fixture("bib_records.json") + EXTRA_RECORDS
by_id = {r["id"]: r for r in records}
res = s6.dedupe(records, title_threshold=THRESHOLD)
print(f"\nĮRAŠŲ: {len(records)}  ->  UNIKALIŲ: {len(res.unique)}  (slenkstis {THRESHOLD:.2f})\n")
for p in res.duplicates:
    a, b = by_id[p.dup_id], by_id[p.canonical_id]
    ratio = difflib.SequenceMatcher(None, s6._normalise_title(a["title"]), s6._normalise_title(b["title"])).ratio()
    print(f"[{p.matched_on:5}] {p.dup_id} pašalintas, paliktas {p.canonical_id}  (pavadinimų panašumas {ratio:.2f})")
    print(f"        {a['title'][:80]}\n        {b['title'][:80]}")
print("\nKAIP KEIČIASI REZULTATAS SU SLENKSČIU")
for t in (0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 1.00):
    n = len(s6.dedupe(records, title_threshold=t).duplicates)
    print(f"  slenkstis {t:.2f}: pašalinta {n}")
print("Pavyzdinėje aibėje tikri dublikatai yra 3. Ties 0.50 pašalinamas 4-as įrašas -- tai jau SKIRTINGAS darbas (klaidingas sujungimas); "
      "ties 0.95 praleidžiamas tikras dublikatas.")

---
## B kelias: parašykite kodą patys

Toliau esantis pratimas skirtas tiems, kas nori išmokti programuoti.

# S06 pratimas

```
S6 pratimas -- deduplikacija.

Užduotis
--------
Faile ``examples/chapter/data/bib_records.json`` yra 18 bibliografinių įrašų,
surinktų iš kelių duomenų bazių. Kai kurie iš jų yra to paties darbo
dublikatai.

1. Įgyvendinkite ``normalise_doi`` -- normalizuokite DOI (mažosios raidės,
   pašalintas ``https://doi.org/`` arba ``http://dx.doi.org/`` prefiksas).
   Naudokite ``str.removeprefix``, NE ``str.lstrip``.
2. Įgyvendinkite ``dedupe`` -- dviejų pakopų kaskadą:
   pirma pagal normalizuotą DOI, paskui pagal neaiškų (fuzzy) pavadinimų
   sutapimą (``difflib.SequenceMatcher``).
3. Deduplikuokite ``bib_records.json``. Turėtų likti 15 unikalių įrašų
   (pašalinami 3 dublikatai: 2 pagal DOI, 1 pagal pavadinimą).
4. Trumpai atsakykite: kurią porą praleistų vien DOI paremtas metodas, o
   kurią -- naivus tikslaus pavadinimo sutapimo metodas?

Paleidimas: ``python -m examples.chapter.s06_deduplication.exercise``
```

1. Paleiskite paruošimo langelį. 2. Užpildykite `TODO` funkcijas pratimo langelyje ir paleiskite jį. 3. Palyginkite su pavyzdiniu sprendimu.

In [ ]:

from __future__ import annotations

from typing import Literal

from pydantic import BaseModel


class DuplicatePair(BaseModel):
    dup_id: str
    canonical_id: str
    matched_on: Literal["doi", "title"]


class DedupeResult(BaseModel):
    unique: list[dict]
    duplicates: list[DuplicatePair]


def normalise_doi(doi: str | None) -> str | None:
    """TODO: normalizuokite DOI arba grąžinkite None."""
    raise NotImplementedError


def dedupe(records: list[dict], *, title_threshold: float = 0.90) -> DedupeResult:
    """TODO: dviejų pakopų deduplikacijos kaskada."""
    raise NotImplementedError


def main() -> None:
    from examples.chapter import toolkit

    records = toolkit.load_fixture("bib_records.json")
    print("S6 deduplication exercise")
    print(f"loaded {len(records)} records; target unique count after dedupe: 15")
    try:
        result = dedupe(records)
    except NotImplementedError:
        print("dedupe() is not implemented yet -- fill it in and re-run.")
        return
    print(f"your dedupe() kept {len(result.unique)} unique records")
    for pair in result.duplicates:
        print(f"  {pair.dup_id} -> {pair.canonical_id} (matched on {pair.matched_on})")


main()

## Pavyzdinis sprendimas

Pažiūrėkite tik pabandę patys. Žemiau esantys testai tikrina **pavyzdinį sprendimą**, ne jūsų kodą.

In [ ]:
!cat examples/chapter/s06_deduplication/solution.py

In [ ]:
!python -m pytest examples/chapter/s06_deduplication -q

In [ ]:
# NEPRIVALOMA: tikri kreipiniai i DKM (sumoka uz naudojimą). Raktą laikykite
# Colab "Secrets" (rakto piktograma kairėje) pavadinimu OPENAI_API_KEY; niekada ne užrašykite kode.
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")